# 进阶实践项目 04：帕金森语音预测中的受试者分组、校准与决策价值

同一受试者可以贡献多段语音记录。按记录随机划分会让同一个人的声音特征同时进入训练和测试，得到看似很高但不能代表新受试者的结果。本项目研究临床预测中最关键的三个问题：独立样本如何定义，预测概率是否可信，模型在某个阈值下是否可能产生实际净收益。


> **实践定位**
>
> 这不是短时间代码竞赛，也不是以最高分数决定完成度的作业。可以只完成数据核对、基线、一个消融实验或一段严谨的失败分析。问题定义、文献依据、方法选择、验证设计、错误解释和下一步实验，重要性高于单一性能数值。
>
> 最终提交由两部分组成：当前 Notebook，以及一份设计报告。设计报告不是代码说明书，而是研究方案说明。参考答案只展示一种能够运行的方案，不代表唯一正确答案，也不意味着其中的模型一定最适合你的目标。


### 可完成的最低范围

读取 UCI Parkinsons 数据，正确提取受试者 ID，比较按记录随机交叉验证与按受试者分组交叉验证，训练逻辑回归或树模型，报告 ROC-AUC、AUPRC、Brier 和校准图，并解释差异。


## 主题背景

分类器首先输出分数或概率，阈值才把它转换为阳性/阴性。区分度评价样本排序，校准评价预测概率与真实频率是否一致，决策曲线把阈值对应的假阳性代价纳入净收益。三者回答不同问题。

预测性能还取决于验证对象。对同一人的重复录音，随机按行验证主要测试模型是否认识已经见过的个体特征；按受试者分组验证才接近面对新人的场景。


## 数据来源与 Kaggle 获取

官方数据来自 UCI Machine Learning Repository：

- 数据页：https://archive.ics.uci.edu/dataset/174/parkinsons
- 数据由 31 名受试者的 195 条语音记录组成，其中 23 名受试者患帕金森病；每行是一条录音，不是一名独立受试者。

Kaggle 中可以搜索 Parkinsons 数据集，或从 UCI 下载 `parkinsons.data` 后创建私人 Kaggle Dataset。文件中的 `name` 类似 `phon_R01_S01_1`，末尾编号是录音序号；受试者 ID 应通过删除最后一个下划线编号得到 `phon_R01_S01`，不能只取第一个字段。


In [ ]:
from pathlib import Path
import pandas as pd, re
paths=list(Path('/kaggle/input').glob('**/parkinsons.data')) if Path('/kaggle/input').exists() else []
if not paths:
    raise FileNotFoundError('请从 UCI 下载 parkinsons.data 并通过 Add Data 挂载。')
df=pd.read_csv(paths[0])
df['subject']=df['name'].str.replace(r'_\d+$','',regex=True)
print('rows:',len(df),'subjects:',df.subject.nunique())
display(df[['name','subject','status']].head(12))
print(df.groupby('subject').status.nunique().value_counts())


## AI 与 Agent 的使用

可以使用 ChatGPT、代码 Agent、Kaggle Notebook Assistant 或其他工具完成资料检索、数据目录检查、代码解释、报错定位、方法比较和报告整理。建议把 AI 当作可审查的协作者，而不是答案来源。

适合交给 AI/Agent 的工作包括：

- 根据实际文件树改写数据读取函数；
- 解释一段代码的输入、输出、shape 和潜在泄漏；
- 比较两种损失、模型或指标的适用条件；
- 根据报错和当前变量状态提出最小修改；
- 搜索论文后整理研究问题、数据、方法、评价和局限；
- 把实验日志整理成设计报告草稿。

所有生成内容都需要核对。论文标题和链接必须打开确认；代码必须逐格运行；数据划分必须用实际 ID 检查；任何“性能提升”都必须由同一测试条件下的结果支持。建议在设计报告末尾记录主要提示词、接受了哪些建议、拒绝了哪些建议以及原因。


## 文献与方法调研

- 原始语音特征研究：Little 等，2007，使用非线性与分形特征识别语音障碍。https://doi.org/10.1186/1475-925X-6-23
- TRIPOD+AI：预测模型研究的透明报告要求。https://www.bmj.com/content/385/bmj-2023-078378
- PROBAST+AI：评估预测模型研究的偏倚风险和适用性。https://www.bmj.com/content/388/bmj-2024-082505
- Decision Curve Analysis：把阈值和误判代价转成净收益。https://pubmed.ncbi.nlm.nih.gov/17099194/

调研时关注样本量、重复测量、特征选择是否在交叉验证内部完成、概率校准、外部验证和目标人群。


## 任务 1：问题定义与数据审计

该数据只能演示语音特征分类方法，不能建立临床可用诊断系统。请说明目标人群、模型输入、输出概率、受试者单位和可能的应用场景。检查每位受试者记录数、类别比例、特征范围、缺失值和相关性。


In [ ]:
# TODO：绘制每位受试者记录数与类别分布。
# TODO：检查缺失、常数列、极端值和高相关特征。


## 任务 2：建立两种验证方案

- 按记录随机交叉验证：作为有意设置的乐观对照；
- 按受试者分组的分层交叉验证：作为主结果。

所有填补、标准化、特征选择和校准都必须在每个训练折内部拟合。建议使用 `Pipeline` 和 `StratifiedGroupKFold`。


In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
# TODO：构造 row_cv 与 subject_cv。
# TODO：用断言检查每折训练和验证 subject 交集为空。


## 任务 3：模型选择

逻辑回归适合作为透明概率基线；XGBoost 或其他树集成可以表达非线性和交互，但在 31 名受试者上更容易过拟合。比较模型时必须使用相同的受试者级折和同一评价代码。

可以选择：逻辑回归；逻辑回归加非线性变换；随机森林；XGBoost/HistGradientBoosting；两种模型的比较。重点是说明为何选它，而不是堆叠模型数量。


In [ ]:
# TODO：在 Pipeline 内实现模型和预处理。
# TODO：生成受试者级 out-of-fold 概率。


## 任务 4：概率评价与阈值

至少报告 ROC-AUC、AUPRC、Brier 和校准曲线。类别不平衡时 AUPRC 更受阳性比例影响。选择一个有解释的阈值，列出真阳性、假阳性、真阴性和假阴性。决策曲线中的阈值代表采取进一步检查或干预所需的最低风险概率，不能脱离真实场景随意解释。


In [ ]:
# TODO：比较 row-level 与 subject-level 验证结果。
# TODO：绘制校准曲线和决策曲线。
# TODO：选择一个阈值并解释误判后果。


## 设计报告是主要提交内容

报告应能够让没有运行 Notebook 的读者理解你的问题、选择和证据。建议正文包含以下内容：

1. **研究问题与动机**：具体要解决什么问题，为什么值得研究，输出将被怎样使用；
2. **数据来源与适用范围**：数据来自体验项目、Kaggle、UCI 或其他公开来源，样本单位、标签、许可、已知偏差和不能代表的人群；
3. **文献调研**：至少阅读两篇原始论文或官方方法文档，说明它们解决的问题、关键方法、评价方式和可借鉴之处；
4. **方案候选与选择理由**：列出考虑过的模型、损失、特征或指标，说明最终选择与算力、样本量、目标和风险之间的关系；
5. **数据划分与验证**：独立样本是谁，怎样避免同一患者、玻片或空间邻域跨集合，哪些指标对应哪些错误；
6. **实现进度与证据**：已经运行的代码、图表、失败现象、异常样本和未完成部分；
7. **结果解释**：结果支持什么、不支持什么，性能较低或没有训练完成也要解释原因；
8. **局限与下一步**：最可能改变结论的限制，以及下一项最值得做的实验；
9. **AI/Agent 使用记录**：主要提示词、采用的建议、人工核查方式和仍未解决的问题。

报告评价重点是思路是否清楚、选择是否有依据、验证是否可信、解释是否诚实。准确率、Dice、AUC 或相关系数只是一部分证据。


### 项目 04 报告还需要回答

- 受试者 ID 怎样从文件名提取，并怎样检查正确性；
- 按记录和按受试者验证分别回答什么问题；
- 区分度、校准和决策曲线为何不能互相替代；
- 数据只有 31 名受试者时，哪些数值最不稳定；
- 迁移到新设备、新语言或新中心前需要什么外部验证。
